In [ ]:
from IPython.display import display, HTML, Markdown
import student_info
display(Markdown("## 商务数据理论与方法——实验2"))
display(HTML(f"<b>姓名：{student_info.NAME}&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;学号：{student_info.STUDENT_ID}</b>"))


>实验主题：数据可视化<br><br>
>实验目的：
>- 掌握matplotlib的基本用法
>- 基于matplotlib和pandas完成数据可视化
    * 趋势可视化
    * 比例可视化 
    * 关联可视化
    * matplotlib可视化示例：https://matplotlib.org/stable/gallery/index.html
    * pandas可视化示例：https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.html#pandas.DataFrame.plot
    
<font color="red">请发送至：<b>teacher\_jin_2\@163.com</b> ，邮件标题和附件标题均为：<b>商务数据理论与方法_学号\_姓名\_实验2</b></font>

### 数据说明
数据集包含了抽样出来的1W用户在一个月时间（11.18~12.18）之内的淘宝移动端行为数据


|字段|字段说明|提取说明|
|----------|:-------------:|------------------:|
|user_id|用户标识|抽样&字段脱敏|
|item_id|商品标识|字段脱敏|
|behavior_name|用户对商品的行为类型| 包括浏览、收藏、加购物车、购买|
|item_category|商品分类标识|字段脱敏|
|time|行为时间|精确到小时级别|

In [ ]:
#实现在同一个cell中输出多个结果
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
#将输出图形嵌入到网页中
%matplotlib inline

# 引入相关的包
#引入相关包
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import gridspec


plt.rcParams['font.sans-serif'] = ['SimHei']  # 替换sans-serif字体
plt.rcParams['axes.unicode_minus'] = False  # 解决坐标轴负数的负号显示问题

### 1. 数据加载与基本信息查看

In [ ]:
#读取数据
data = pd.read_csv("user_action.csv",usecols=[1,2,3,4,5],encoding="utf-8")

In [ ]:
#数据备份
dt = data.copy()

In [ ]:
#数据抽样查看
dt.sample(10)

In [ ]:
#数据信息查看
dt.info()

In [ ]:
# 查看整体数据规模，包括整体数据、用户数、商品数、商品类别数等
summary={
    "整体数据":dt.shape[0],
    "用户数":len(set(dt["user_id"])),
    "商品数":len(set((dt["item_id"]))),
    "商品类别":len(set(dt["item_category"]))
}

summary

### 2. 数据处理

> 由于数据的最小统计时间单位是小时，后续的分析可以从小时、天等时间跨度进行数据分析，考虑将时间进行分割处理

In [ ]:
# 先将行为时间time分割天(date)和小时(hour)，以便后续的进一步分析
dt["date"] = dt["time"].map(lambda x:x.split(" ")[0])
dt["hour"] = dt["time"].map(lambda x:x.split(" ")[1])
dt.sample(10)

In [ ]:
# 查看数据情况
dt.info()

In [ ]:
# 数据类型转换
dt['date'] = pd.to_datetime(dt['date'])
dt['hour'] = dt['hour'].astype('int64')

In [ ]:
# 查看数据类型
dt.dtypes

### 3 数据可视化分析

#### 3.1 趋势可视化
> 使用PV（页面访问量）和UV（独立访客数）指标，分析访问量和访问用户的趋势和规律。用可视化工具绘制相应的图表更好地展示数据，进而找出高峰期及用户访问偏好。同时，我们发现提供的数据有“双十二”这一特殊日期，因此可针对12月12日这一天进行具体分析，以便更好地规划营销活动。

* PV（页面访问量）即Page View，用户每次对网站的访问均被记录，用户对同一页面的多次访问，是访问量的累计。
* UV（独立访客数）即Unique Visitor，访问网站的一台电脑客户端为一个访客，根据IP地址来区分访客数。

##### 3.1.1 PV趋势可视化

（1）PV统计

In [ ]:
#### PV统计
pv_daily = dt.groupby("date")['user_id'].count().reset_index().rename(columns={'user_id':'pv_daily'})
pv_daily

（2）PV趋势可视化

In [ ]:
#可视化代码
#参考;https://matplotlib.org/stable/users/explain/quick_start.html#sphx-glr-users-explain-quick-start-py
fig = plt.figure(figsize=(15,5))#创建画布
axes = fig.add_subplot(1,1,1)#创建坐标系
axes.set_title("PV趋势分析",fontsize=15,color="blue")
axes.set_xlabel("时间",fontsize=13,color="red")
axes.set_ylabel("PV值",fontsize=13,color="red")
axes.plot(pv_daily["date"],pv_daily["pv_daily"])

##### 3.1.2 UV趋势分析
（1）UV统计

In [ ]:
#### UV统计

uv_daily = dt.groupby("date")['user_id'].nunique().reset_index().rename(columns={'user_id':'uv_daily'})
uv_daily

（2）UV趋势可视化

In [ ]:
### 可视化代码

fig = plt.figure(figsize=(15,5))#创建画布
axes = fig.add_subplot(1,1,1)#创建坐标系
axes.set_title("UV趋势分析",fontsize=15,color="blue")
axes.set_xlabel("时间",fontsize=13,color="red")
axes.set_ylabel("UV值",fontsize=13,color="red")
axes.plot(uv_daily["date"],uv_daily["uv_daily"])



（3）PV和UV趋势对比分析

In [ ]:
### 将PV和UV的趋势放在一个坐标系中——多折线图
### 参考：https://matplotlib.org/stable/users/explain/quick_start.html#sphx-glr-users-explain-quick-start-py






#### 3.2 比例可视化

##### 3.2.1 统计销量最好商品的浏览、收藏、加购物车和购买的占比情况

（1）销量最高的商品识别

In [ ]:
# 获取用户行为数据
tmp = dt[dt["behavior_name"]=="购买"].groupby(["item_id"])["user_id"].count() #统计所有商品的购买数量
best_sale_item_id = tmp.sort_values(ascending=False).index[0] #得到销售最好的商品ID

（2）统计销量最好商品的浏览、收藏、加购物车和购买数量

In [ ]:
tmp = dt[dt["item_id"]==best_sale_item_id]
# 获取用户行为数据
behavior_statc = tmp.groupby(['behavior_name'])['user_id'].count()
behavior_statc

（3）比例可视化

In [ ]:
#比例代码
#参考;https://matplotlib.org/stable/users/explain/quick_start.html#sphx-glr-users-explain-quick-start-py
fig = plt.figure(figsize=(10,5))#创建画布

ax1 = fig.add_subplot(1,2,1)#创建坐标系
ax1.set_title("PV的构成分析",fontsize=12,color="blue")
ax1.set_ylabel("数量",fontsize=10,color="red")
ax1.set_xlabel("操作类型",fontsize=10,color="red")
ax1.bar(behavior_statc.index,behavior_statc.values)

ax2 = fig.add_subplot(1,2,2)#创建坐标系
ax2.set_title("PV的比例分析",fontsize=12,color="blue")
ax2.pie(behavior_statc.values,labels=behavior_statc.index,autopct='%1.1f%%')


##### 3.2.2 统计浏览次数最多商品的浏览、收藏、加购物车和购买的占比情况

（1）浏览最多的商品识别

In [ ]:
# show me you code



（2）浏览最多商品的浏览、收藏、加购物车和购买数量

In [ ]:
# show me you code



（3）比例可视化

In [ ]:
# show me you code




#### 3.3 关联可视化

> 随机选择100件商品，绘制多个散点图来分析浏览、收藏、加购物车与购买之间的关联关系

> 散点图的绘制参考：https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.scatter.html#matplotlib.axes.Axes.scatter

In [ ]:
# show me your code









### Enjoy